# Курсовой проект по дисциплине «Технологии обработки больших данных»

**Тема:** «Исследование предпочтений в музыке на основе потоковых и текстовых данных»  
**Этап:** 2.3. Подготовка, очистка и слияние разнородных данных

На этом этапе выполняется реальная подготовка данных из источников, выбранных в п. 2.2: Spotify Top 200, MusicBrainz XML, Last.fm JSON API и The Guardian. В соответствии с методическими указаниями проводится первичный анализ, обработка пропусков и дубликатов, определение аналитических шкал и ролей признаков, аннотирование и векторизация текста, нормализация общих ключей и слияние данных.

Для API используется ограниченная выборка наиболее популярных объектов. Это позволяет воспроизводимо выполнить этап и не создавать избыточную нагрузку на открытые сервисы.

## 1. Подключение библиотек и параметры

Для основного CSV ключи не требуются. MusicBrainz также работает без API-ключа. Для Last.fm и The Guardian нужны бесплатные ключи. Они не записываются в публичный репозиторий.

Если ключи уже заданы как переменные окружения LASTFM_API_KEY и GUARDIAN_API_KEY, ноутбук использует их автоматически. Если ключа нет, соответствующий API-блок будет пропущен, а остальные части ноутбука продолжат работать.

In [2]:
import os
import re
import time
import html
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import xml.etree.ElementTree as ET

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import sparse

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

SPOTIFY_URL = (
    "https://raw.githubusercontent.com/younver/"
    "spotify-top-200-dataset/main/spotify-top-200-dataset.csv"
)

LASTFM_API_KEY = os.getenv("LASTFM_API_KEY", "")
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY", "")

MB_ARTIST_LIMIT = 10
LASTFM_TRACK_LIMIT = 30
GUARDIAN_ARTIST_LIMIT = 8
GUARDIAN_ARTICLES_PER_ARTIST = 5

print("Last.fm key:", "задан" if LASTFM_API_KEY else "не задан")
print("Guardian key:", "задан" if GUARDIAN_API_KEY else "не задан")


/Users/Arina/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Last.fm key: задан
Guardian key: задан


### Как временно задать ключи в Colab

Перед повторным запуском API-ячеек можно выполнить отдельную временную ячейку:

    import os
    os.environ["LASTFM_API_KEY"] = "ВАШ_КЛЮЧ"
    os.environ["GUARDIAN_API_KEY"] = "ВАШ_КЛЮЧ"

Эту ячейку с реальными значениями ключей не нужно сохранять в публичный ноутбук.

## 2. Загрузка основного реляционного набора Spotify Top 200

Основной набор содержит недельные наблюдения глобального Spotify Top-200 за 2017–2021 годы. В исходном репозитории заявлено 74 661 строка и 40 столбцов. Сначала проверим фактический размер, типы данных, первые строки и пропуски.

In [3]:
spotify_raw = pd.read_csv(SPOTIFY_URL, sep=";", low_memory=False)

print("Размер исходного набора:", spotify_raw.shape)
print("\nПервые 5 строк:")
display(spotify_raw.head())

print("\nТипы данных:")
display(
    spotify_raw.dtypes.rename("dtype")
    .astype(str)
    .to_frame()
)

print("\nПропуски:")
missing_spotify = (
    spotify_raw.isna()
    .sum()
    .sort_values(ascending=False)
    .rename("missing")
    .to_frame()
)
missing_spotify["share_%"] = (
    missing_spotify["missing"] / len(spotify_raw) * 100
).round(2)
display(missing_spotify.head(20))

print("Полных дубликатов:", spotify_raw.duplicated().sum())


Размер исходного набора: (74660, 40)

Первые 5 строк:


,track_id,track_name,track_popularity,track_number,album_id,album_name,album_img,album_type,album_label,album_track_number,album_popularity,artist_num,artist_names,artist_id,artist_name,artist_img,artist_followers,artist_popularity,artist_genres,rank,week,collab,explicit,release_date,danceability,energy,key,mode,time_signature,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,pivot,streams,track_index
0,5aAx2yezTd8zXrkmtKl66Z,Starboy,0,1,09fggMHib4YkOtwQNXEBII,Starboy,https://i.scdn.co/image/ab67616d0000b2730c8599cbde51245c128bcea9,album,Universal Music Group,1,0,2,"The Weeknd, Daft Punk",1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,https://i.scdn.co/image/ab6761610000e5ebb5f9e28219c169fd4b9e8379,51215861,94,"canadian contemporary r&b, canadian pop, pop",1,1/6/2017,True,True,2016-11-25,0.681,0.594,7,1,4,-7.028,0.2820,0.165,0.000003,0.134,0.535,186.054,230453,0,25734078,1
1,5aAx2yezTd8zXrkmtKl66Z,Starboy,0,1,09fggMHib4YkOtwQNXEBII,Starboy,https://i.scdn.co/image/ab67616d0000b2730c8599cbde51245c128bcea9,album,Universal Music Group,1,0,2,"The Weeknd, Daft Punk",4tZwfgrHOc3mvqYlEYSvVi,Daft Punk,https://i.scdn.co/image/ab6761610000e5eb96d66c60658005885d1135ce,8953122,81,"electro, filter house",1,1/6/2017,True,True,2016-11-25,0.681,0.594,7,1,4,-7.028,0.2820,0.165,0.000003,0.134,0.535,186.054,230453,1,25734078,1
2,7BKLCZ1jbUBVqRi2FVlTVw,Closer,84,1,0rSLgV8p5FzfnqlEk4GzxE,Closer,https://i.scdn.co/image/ab67616d0000b273495ce6da9aeb159e94eaa453,single,Disruptor Records/Columbia,1,71,2,"The Chainsmokers, Halsey",69GGBxA162lTqCwzJG5jLp,The Chainsmokers,https://i.scdn.co/image/ab6761610000e5eb3c02f4fb4cc9187c488afd50,19562506,81,"dance pop, edm, electropop, pop, pop dance, tropical house",2,1/6/2017,True,False,2016-07-29,0.748,0.524,8,1,4,-5.599,0.0338,0.414,0.000000,0.111,0.661,95.010,244960,0,23519705,2
3,7BKLCZ1jbUBVqRi2FVlTVw,Closer,84,1,0rSLgV8p5FzfnqlEk4GzxE,Closer,https://i.scdn.co/image/ab67616d0000b273495ce6da9aeb159e94eaa453,single,Disruptor Records/Columbia,1,71,2,"The Chainsmokers, Halsey",26VFTg2z8YR0cCuwLzESi2,Halsey,https://i.scdn.co/image/ab6761610000e5ebd707e1c5177614c4ec95a06c,18598273,84,"dance pop, electropop, etherpop, indie poptimism, pop",2,1/6/2017,True,False,2016-07-29,0.748,0.524,8,1,4,-5.599,0.0338,0.414,0.000000,0.111,0.661,95.010,244960,1,23519705,2
4,5knuzwU65gJK7IF5yJsuaW,Rockabye (feat. Sean Paul & Anne-Marie),75,1,3meZFplbMmji648oWUNEfQ,Rockabye (feat. Sean Paul & Anne-Marie),https://i.scdn.co/image/ab67616d0000b2731431c3bdf16aa99f71799d95,single,Atlantic Records UK,1,62,3,"Clean Bandit, Sean Paul, Anne-Marie",6MDME20pz9RveH9rEXvrOM,Clean Bandit,https://i.scdn.co/image/ab6761610000e5eb7a487027eb0c10af725d5410,4943046,76,"dance pop, edm, pop, pop dance, post-teen pop, tropical house, uk dance, uk funky, uk pop",3,1/6/2017,True,False,2016-10-21,0.720,0.763,9,0,4,-4.068,0.0523,0.406,0.000000,0.180,0.742,101.965,251088,0,21216399,3



Типы данных:


,dtype
track_id,object
track_name,object
track_popularity,int64
track_number,int64
album_id,object
album_name,object
album_img,object
album_type,object
album_label,object
album_track_number,int64



Пропуски:


,missing,share_%
artist_genres,997,1.34
artist_img,125,0.17
track_name,5,0.01
album_name,5,0.01
album_img,5,0.01
album_label,5,0.01
artist_names,5,0.01
track_id,0,0.00
speechiness,0,0.00
key,0,0.00


Полных дубликатов: 0


## 3. Физический смысл и аналитическая роль используемых признаков

Для проверки гипотезы используются не все 40 полей, а признаки, описывающие популярность музыкального контента и возможные факторы влияния.

| Признак | Физический смысл | Аналитическая роль |
|---|---|---|
| track_id | идентификатор трека Spotify | ID / ключ |
| track_name | название композиции | категория / ключ слияния |
| artist_name | исполнитель | категория / ключ слияния |
| artist_genres | жанры исполнителя | фактор |
| album_type | тип релиза | фактор |
| explicit | наличие explicit-контента | фактор |
| collab | наличие нескольких исполнителей | фактор |
| week | неделя нахождения в чарте | время |
| release_date | дата релиза | время / фактор |
| streams | число недельных прослушиваний | основной показатель популярности |
| rank | место в чарте | показатель популярности |
| track_popularity | индекс популярности трека Spotify | показатель популярности |
| artist_popularity | индекс популярности исполнителя | фактор |
| artist_followers | число подписчиков исполнителя | фактор |
| danceability | танцевальность | фактор |
| energy | энергичность | фактор |
| valence | позитивность звучания | фактор |
| tempo | темп композиции | фактор |
| duration | длительность | фактор |
| listeners | число слушателей Last.fm | внешний показатель интереса |
| playcount | число проигрываний Last.fm | внешний показатель интереса |
| country | страна исполнителя MusicBrainz | фактор |
| type | тип исполнителя MusicBrainz | фактор |
| bodyText | текст статьи Guardian | неструктурированный контент |
| article_count | количество статей | внешний информационный фактор |

## 4. Аналитические шкалы

По типологии, используемой в курсовом проекте, признаки распределены по восьми видам шкал.

| Тип шкалы | Примеры в проекте | Обоснование |
|---|---|---|
| Номинальная (ID) | track_id, artist_id, album_id, mbid | идентификаторы, над которыми не выполняются арифметические операции |
| Контентная | bodyText, исходный XML и JSON | слабоструктурированный или неструктурированный контент |
| Категориальная | artist_name, artist_genres, album_type, country | конечное множество категорий без порядка |
| Дихотомическая | explicit, collab, mode | два состояния: да/нет |
| Порядковая | rank | значения упорядочены, но разность между соседними рангами не является равным интервалом |
| Интервальная | week, release_date, webPublicationDate, track_popularity | разности значений интерпретируемы, нулевая точка условна |
| Отношений | duration, tempo | естественный ноль, отношения значений имеют смысл |
| Абсолютная | streams, listeners, playcount, artist_followers, word_count, article_count | результат непосредственного подсчета с фиксированной единицей |

## 5. Очистка Spotify

В исходном наборе коллаборации могут быть представлены несколькими строками: по одной на каждого исполнителя. Поле pivot отмечает дополнительные строки. Для анализа на уровне «трек–неделя» оставляется основная строка pivot = 0. Также приводятся типы, обрабатываются пропуски и создаются нормализованные ключи для дальнейшего слияния.

In [4]:
def normalize_key(value):
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9а-яё]+", " ", value, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", value).strip()

def to_boolean(series):
    def convert(value):
        if pd.isna(value):
            return pd.NA
        if isinstance(value, bool):
            return value
        text = str(value).strip().lower()
        if text in {"1", "true", "yes"}:
            return True
        if text in {"0", "false", "no"}:
            return False
        return pd.NA
    return series.map(convert).astype("boolean")

spotify = spotify_raw.copy()

# Даты
for col in ["week", "release_date"]:
    if col in spotify.columns:
        spotify[col] = pd.to_datetime(spotify[col], errors="coerce")

# Числовые признаки
numeric_cols = [
    "track_popularity", "album_popularity", "artist_followers",
    "artist_popularity", "rank", "streams", "danceability",
    "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo", "duration"
]

for col in numeric_cols:
    if col in spotify.columns:
        spotify[col] = pd.to_numeric(spotify[col], errors="coerce")

# Логические признаки
for col in ["explicit", "collab", "mode", "pivot"]:
    if col in spotify.columns:
        spotify[col] = to_boolean(spotify[col])

# Оставляем основную строку трека при разбиении коллабораций
if "pivot" in spotify.columns:
    spotify = spotify[(spotify["pivot"].isna()) | (~spotify["pivot"])].copy()

# Удаляем строки без критически важных полей
required = [c for c in ["track_name", "artist_name", "week", "streams"] if c in spotify.columns]
spotify = spotify.dropna(subset=required)

# Удаляем повтор трек-неделя после фильтра pivot
dedup_keys = [c for c in ["track_id", "week"] if c in spotify.columns]
if dedup_keys:
    spotify = spotify.drop_duplicates(subset=dedup_keys, keep="first")

# Пропуски категорий
for col in ["artist_genres", "album_type", "album_label"]:
    if col in spotify.columns:
        spotify[col] = spotify[col].fillna("unknown")

# Медианная импутация аудиопризнаков
audio_cols = [
    c for c in [
        "danceability", "energy", "loudness", "speechiness",
        "acousticness", "instrumentalness", "liveness",
        "valence", "tempo", "duration"
    ]
    if c in spotify.columns
]

for col in audio_cols:
    spotify[col] = spotify[col].fillna(spotify[col].median())

# Нормализованные ключи
spotify["artist_key"] = spotify["artist_name"].map(normalize_key)
spotify["track_key"] = spotify["track_name"].map(normalize_key)

print("Размер после очистки:", spotify.shape)
print("Удалено строк:", len(spotify_raw) - len(spotify))
print("Полных дубликатов после очистки:", spotify.duplicated().sum())

display(spotify.head())


Размер после очистки: (44195, 42)
Удалено строк: 30465
Полных дубликатов после очистки: 0


,track_id,track_name,track_popularity,track_number,album_id,album_name,album_img,album_type,album_label,album_track_number,album_popularity,artist_num,artist_names,artist_id,artist_name,artist_img,artist_followers,artist_popularity,artist_genres,rank,week,collab,explicit,release_date,danceability,energy,key,mode,time_signature,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,pivot,streams,track_index,artist_key,track_key
0,5aAx2yezTd8zXrkmtKl66Z,Starboy,0,1,09fggMHib4YkOtwQNXEBII,Starboy,https://i.scdn.co/image/ab67616d0000b2730c8599cbde51245c128bcea9,album,Universal Music Group,1,0,2,"The Weeknd, Daft Punk",1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,https://i.scdn.co/image/ab6761610000e5ebb5f9e28219c169fd4b9e8379,51215861,94,"canadian contemporary r&b, canadian pop, pop",1,2017-01-06,True,True,2016-11-25,0.681,0.594,7,True,4,-7.028,0.2820,0.1650,0.000003,0.134,0.5350,186.054,230453,False,25734078,1,the weeknd,starboy
2,7BKLCZ1jbUBVqRi2FVlTVw,Closer,84,1,0rSLgV8p5FzfnqlEk4GzxE,Closer,https://i.scdn.co/image/ab67616d0000b273495ce6da9aeb159e94eaa453,single,Disruptor Records/Columbia,1,71,2,"The Chainsmokers, Halsey",69GGBxA162lTqCwzJG5jLp,The Chainsmokers,https://i.scdn.co/image/ab6761610000e5eb3c02f4fb4cc9187c488afd50,19562506,81,"dance pop, edm, electropop, pop, pop dance, tropical house",2,2017-01-06,True,False,2016-07-29,0.748,0.524,8,True,4,-5.599,0.0338,0.4140,0.000000,0.111,0.6610,95.010,244960,False,23519705,2,the chainsmokers,closer
4,5knuzwU65gJK7IF5yJsuaW,Rockabye (feat. Sean Paul & Anne-Marie),75,1,3meZFplbMmji648oWUNEfQ,Rockabye (feat. Sean Paul & Anne-Marie),https://i.scdn.co/image/ab67616d0000b2731431c3bdf16aa99f71799d95,single,Atlantic Records UK,1,62,3,"Clean Bandit, Sean Paul, Anne-Marie",6MDME20pz9RveH9rEXvrOM,Clean Bandit,https://i.scdn.co/image/ab6761610000e5eb7a487027eb0c10af725d5410,4943046,76,"dance pop, edm, pop, pop dance, post-teen pop, tropical house, uk dance, uk funky, uk pop",3,2017-01-06,True,False,2016-10-21,0.720,0.763,9,False,4,-4.068,0.0523,0.4060,0.000000,0.180,0.7420,101.965,251088,False,21216399,3,clean bandit,rockabye feat sean paul anne marie
7,4pdPtRcBmOSQDlJ3Fk945m,Let Me Love You,2,13,55bbXORm6ZrVq52zfZnxBf,Encore,https://i.scdn.co/image/ab67616d0000b2735045de6786e0af096de73ed4,album,Universal Music Group,13,0,2,"DJ Snake, Justin Bieber",540vIaP2JwjQb9dm3aArA4,DJ Snake,https://i.scdn.co/image/ab6761610000e5ebf2855e25f1d9c8a20bcc85ae,8137850,78,"dance pop, edm, electronic trap, pop, pop dance, pop rap",4,2017-01-06,True,False,2016-08-05,0.476,0.718,8,True,4,-5.309,0.0576,0.0784,0.000010,0.122,0.1420,199.864,205946,False,19852704,4,dj snake,let me love you
9,3NdDpSvN911VPGivFlV5d0,"I Don’t Wanna Live Forever (Fifty Shades Darker) - From ""Fifty Shades Darker (Original Motion Picture Soundtrack)""",0,1,5MxXY7DbFMUiHFTPUabgJJ,I Don’t Wanna Live Forever (Fifty Shades Darker),https://i.scdn.co/image/ab67616d0000b2732ae921f082ae5db4d3da818f,single,Universal Music Group,1,0,2,"ZAYN, Taylor Swift",5ZsFI1h6hIdQRw2ti0hz81,ZAYN,https://i.scdn.co/image/ab6761610000e5eb9fbf7133dfc04d4cd44ccd36,21433236,77,"dance pop, pop, post-teen pop, uk pop",5,2017-01-06,True,False,2016-12-09,0.735,0.451,0,True,4,-8.374,0.0585,0.0631,0.000013,0.325,0.0862,117.973,245200,False,18316326,5,zayn,i don t wanna live forever fifty shades darker from fifty shades darker original motion picture soundtrack


### Результат очистки Spotify

После подготовки одна строка соответствует наблюдению композиции за определенную неделю. Нормализованные artist_key и track_key будут использоваться только как технические ключи слияния; исходные названия сохраняются.

## 6. Выбор объектов для API-источников

Для открытых API используется не весь список объектов, а наиболее значимые по суммарному числу стримов. Это сокращает число запросов, соблюдает ограничения внешних сервисов и при этом сохраняет объекты, наиболее важные для исследования популярности.

In [5]:
top_artists = (
    spotify.groupby(["artist_key", "artist_name"], as_index=False)["streams"]
    .sum()
    .sort_values("streams", ascending=False)
)

top_tracks = (
    spotify.groupby(
        ["track_key", "track_name", "artist_key", "artist_name"],
        as_index=False
    )["streams"]
    .sum()
    .sort_values("streams", ascending=False)
)

print("Топ исполнителей:")
display(top_artists.head(10))

print("Топ треков:")
display(top_tracks.head(10))


Топ исполнителей:


,artist_key,artist_name,streams
603,post malone,Post Malone,13565225335
215,ed sheeran,Ed Sheeran,11279402393
203,drake,Drake,9075103253
83,billie eilish,Billie Eilish,8454013396
49,ariana grande,Ariana Grande,8106074363
798,xxxtentacion,XXXTENTACION,6958468654
66,bad bunny,Bad Bunny,6113728945
740,the weeknd,The Weeknd,5695764620
355,juice wrld,Juice WRLD,5502518229
207,dua lipa,Dua Lipa,5268421383


Топ треков:


,track_key,track_name,artist_key,artist_name,streams
2756,shape of you,Shape of You,ed sheeran,Ed Sheeran,2678476483
741,dance monkey,Dance Monkey,tones and i,Tones And I,2131900361
421,blinding lights,Blinding Lights,the weeknd,The Weeknd,2097819581
2986,sunflower spider man into the spider verse,Sunflower - Spider-Man: Into the Spider-Verse,post malone,Post Malone,1840864152
2883,someone you loved,Someone You Loved,lewis capaldi,Lewis Capaldi,1812429313
2722,se orita,Señorita,shawn mendes,Shawn Mendes,1771172093
280,bad guy,bad guy,billie eilish,Billie Eilish,1728907464
2392,perfect,Perfect,ed sheeran,Ed Sheeran,1642675210
350,believer,Believer,imagine dragons,Imagine Dragons,1636172918
1894,lucid dreams,Lucid Dreams,juice wrld,Juice WRLD,1578807899


## 7. MusicBrainz: получение и разбор XML

MusicBrainz используется как иерархический XML-источник. Для небольшой выборки топ-исполнителей выполняется поиск наиболее подходящего объекта. Из XML извлекаются идентификатор MusicBrainz, имя, тип исполнителя, страна, пол, период активности и число альтернативных имен.

Для соблюдения рекомендаций MusicBrainz между запросами добавлена пауза.

In [6]:
MB_NS = {"m": "http://musicbrainz.org/ns/mmd-2.0#"}
MB_HEADERS = {
    "User-Agent": "TOBD-course-project/1.0 (educational research)"
}

def musicbrainz_artist(name):
    response = requests.get(
        "https://musicbrainz.org/ws/2/artist/",
        params={"query": f'artist:"{name}"', "limit": 1},
        headers=MB_HEADERS,
        timeout=30
    )
    response.raise_for_status()
    root = ET.fromstring(response.text)
    artist = root.find(".//m:artist", MB_NS)

    if artist is None:
        return None

    life_span = artist.find("m:life-span", MB_NS)
    aliases = artist.findall("m:alias-list/m:alias", MB_NS)

    return {
        "artist_key": normalize_key(name),
        "mbid": artist.attrib.get("id", ""),
        "mb_type": artist.attrib.get("type", ""),
        "mb_name": artist.findtext("m:name", default="", namespaces=MB_NS),
        "country": artist.findtext("m:country", default="", namespaces=MB_NS),
        "gender": artist.findtext("m:gender", default="", namespaces=MB_NS),
        "active_begin": (
            life_span.findtext("m:begin", default="", namespaces=MB_NS)
            if life_span is not None else ""
        ),
        "active_end": (
            life_span.findtext("m:end", default="", namespaces=MB_NS)
            if life_span is not None else ""
        ),
        "alias_count": len(aliases)
    }

mb_rows = []

for artist_name in top_artists["artist_name"].head(MB_ARTIST_LIMIT):
    try:
        row = musicbrainz_artist(artist_name)
        if row:
            mb_rows.append(row)
    except Exception as exc:
        print("MusicBrainz:", artist_name, "->", exc)
    time.sleep(1.05)

musicbrainz_df = pd.DataFrame(mb_rows)

print("Получено объектов MusicBrainz:", len(musicbrainz_df))
display(musicbrainz_df.head(10))


Получено объектов MusicBrainz: 10


,artist_key,mbid,mb_type,mb_name,country,gender,active_begin,active_end,alias_count
0,post malone,b1e26560-60e5-4236-bbdb-9aa5a8d5ee19,Person,Post Malone,US,male,1995-07-04,,4
1,ed sheeran,b8a7c51f-362c-4dcb-a259-bc6e0095f0a6,Person,Ed Sheeran,GB,male,1991-02-17,,2
2,drake,9fff2f8a-21e6-47de-a2b8-7f449929d43f,Person,Drake,CA,male,1986-10-24,,4
3,billie eilish,f4abc0b5-3f7a-4eff-8f78-ac078dbce533,Person,Billie Eilish,US,female,2001-12-18,,25
4,ariana grande,f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387,Person,Ariana Grande,US,female,1993-06-26,,12
5,xxxtentacion,61af87f4-16ee-4431-8504-cc06187079fb,Person,XXXTENTACION,US,male,1998-01-23,2018-06-18,12
6,bad bunny,89aa5ecb-59ad-46f5-b3eb-2d424e941f19,Person,Bad Bunny,PR,male,1994-03-10,,1
7,the weeknd,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,Person,The Weeknd,CA,male,1990-02-16,,7
8,juice wrld,4e4ebde4-0c56-4dec-844b-6c73adcdd92d,Person,Juice WRLD,US,male,1998-12-02,2019-12-08,6
9,dua lipa,6f1a58bf-9b1b-49cf-a44a-6cefad7ae04f,Person,Dua Lipa,GB,female,1995-08-22,,5


## 8. Last.fm: подготовка JSON-данных

Для треков из Spotify запрашивается метод track.getInfo. Он позволяет получить дополнительные числовые показатели listeners и playcount, длительность, альбом и пользовательские теги. Слияние с Spotify выполняется по нормализованной паре «исполнитель + трек».

Если API-ключ не задан, создается пустая таблица нужной структуры, чтобы остальные ячейки ноутбука продолжали выполняться.

In [7]:
LASTFM_COLUMNS = [
    "artist_key", "track_key", "lfm_mbid", "lfm_duration",
    "listeners", "playcount", "lfm_album", "lfm_tags"
]

def lastfm_track_info(artist, track):
    response = requests.get(
        "https://ws.audioscrobbler.com/2.0/",
        params={
            "method": "track.getInfo",
            "api_key": LASTFM_API_KEY,
            "artist": artist,
            "track": track,
            "autocorrect": 1,
            "format": "json"
        },
        headers={"User-Agent": "TOBD-course-project/1.0"},
        timeout=30
    )
    response.raise_for_status()
    data = response.json()

    if "track" not in data:
        return None

    t = data["track"]
    tags = [
        x.get("name", "")
        for x in t.get("toptags", {}).get("tag", [])
    ]

    return {
        "artist_key": normalize_key(t.get("artist", {}).get("name", artist)),
        "track_key": normalize_key(t.get("name", track)),
        "lfm_mbid": t.get("mbid", ""),
        "lfm_duration": pd.to_numeric(t.get("duration"), errors="coerce"),
        "listeners": pd.to_numeric(t.get("listeners"), errors="coerce"),
        "playcount": pd.to_numeric(t.get("playcount"), errors="coerce"),
        "lfm_album": t.get("album", {}).get("title", ""),
        "lfm_tags": ", ".join(tags)
    }

lastfm_rows = []

if LASTFM_API_KEY:
    if "top_tracks" not in globals():
        raise RuntimeError(
            "Переменная top_tracks не создана. "
            "Сначала выполните ячейки загрузки и очистки Spotify сверху вниз."
        )

    sample_tracks = top_tracks.head(LASTFM_TRACK_LIMIT)

    for _, item in sample_tracks.iterrows():
        try:
            row = lastfm_track_info(item["artist_name"], item["track_name"])
            if row:
                lastfm_rows.append(row)
        except Exception as exc:
            print("Last.fm:", item["artist_name"], "-", item["track_name"], "->", exc)
        time.sleep(0.15)

    lastfm_df = pd.DataFrame(lastfm_rows, columns=LASTFM_COLUMNS)
else:
    print("LASTFM_API_KEY не задан: запросы Last.fm пропущены.")
    lastfm_df = pd.DataFrame(columns=LASTFM_COLUMNS)

print("Получено объектов Last.fm:", len(lastfm_df))
display(lastfm_df.head())


Получено объектов Last.fm: 30


,artist_key,track_key,lfm_mbid,lfm_duration,listeners,playcount,lfm_album,lfm_tags
0,ed sheeran,shape of you,043506eb-40ac-43a2-99d0-c789b6985bee,233000,1444862,12559108,÷ (Deluxe),"pop, dancehall, 2017, Ed Sheeran, best of 2017"
1,tones and i,dance monkey,034db3f5-2bc5-4724-bbb1-ecbf86e3558e,209000,784227,7600043,Housework Songs,
2,the weeknd,blinding lights,08dec059-f47e-4720-b363-017ac940eebc,200000,2448729,39655528,Now 35,"synthwave, synthpop, pop, 2019, 2010s"
3,post malone,sunflower spider man into the spider verse,a66aa513-1476-4a25-bcaa-9f465cde285a,158000,1888172,27571861,Scrobbling Makes Me Horney,"Hip-Hop, pop, rnb, 2018, 2010s"
4,lewis capaldi,someone you loved,03ec8afa-14ad-4cdf-8c36-386ea075676c,182000,1102721,12730559,Best of Electronic Music,"2010s, pop, 2019, 2018, lewis capaldi"


## 9. The Guardian: сбор и аннотирование неструктурированного текста

The Guardian используется как источник неструктурированных текстов. Статьи запрашиваются по именам наиболее популярных исполнителей в разделе music.

Аннотирование выполняется путем создания дополнительных признаков:
- длина текста;
- количество слов;
- количество тегов;
- число упоминаний исполнителя;
- наличие имени исполнителя в заголовке.

После этого очищенный текст будет векторизован методом TF-IDF.

In [8]:
GUARDIAN_COLUMNS = [
    "artist_key", "artist_name", "article_id", "title", "publication_date",
    "author", "body_text", "word_count_api", "tag_count",
    "text_length", "word_count", "artist_mention_count",
    "artist_in_title"
]

def clean_article_text(text):
    text = html.unescape(str(text or ""))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^A-Za-zА-Яа-яЁё0-9'\-\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip().lower()

def guardian_articles(artist):
    response = requests.get(
        "https://content.guardianapis.com/search",
        params={
            "q": artist,
            "section": "music",
            "from-date": "2017-01-01",
            "to-date": "2025-12-31",
            "page-size": GUARDIAN_ARTICLES_PER_ARTIST,
            "order-by": "relevance",
            "show-fields": "headline,byline,bodyText,wordcount",
            "show-tags": "keyword",
            "api-key": GUARDIAN_API_KEY
        },
        timeout=30
    )
    response.raise_for_status()
    return response.json()["response"]["results"]

guardian_rows = []

if GUARDIAN_API_KEY:
    if "top_artists" not in globals():
        raise RuntimeError(
            "Переменная top_artists не создана. "
            "Сначала выполните ячейки загрузки и очистки Spotify сверху вниз."
        )

    for artist_name in top_artists["artist_name"].head(GUARDIAN_ARTIST_LIMIT):
        try:
            results = guardian_articles(artist_name)

            for item in results:
                fields = item.get("fields", {})
                tags = item.get("tags", [])
                title = item.get("webTitle", "")
                body = fields.get("bodyText", "")
                clean = clean_article_text(body)
                artist_norm = artist_name.lower()

                guardian_rows.append({
                    "artist_key": normalize_key(artist_name),
                    "artist_name": artist_name,
                    "article_id": item.get("id", ""),
                    "title": title,
                    "publication_date": pd.to_datetime(
                        item.get("webPublicationDate"),
                        errors="coerce"
                    ),
                    "author": fields.get("byline", ""),
                    "body_text": clean,
                    "word_count_api": pd.to_numeric(
                        fields.get("wordcount"),
                        errors="coerce"
                    ),
                    "tag_count": len(tags),
                    "text_length": len(clean),
                    "word_count": len(clean.split()),
                    "artist_mention_count": clean.count(artist_norm),
                    "artist_in_title": artist_norm in title.lower()
                })

        except Exception as exc:
            print("Guardian:", artist_name, "->", exc)

    guardian_df = pd.DataFrame(guardian_rows, columns=GUARDIAN_COLUMNS)
else:
    print("GUARDIAN_API_KEY не задан: запросы The Guardian пропущены.")
    guardian_df = pd.DataFrame(columns=GUARDIAN_COLUMNS)

print("Получено статей:", len(guardian_df))
display(guardian_df.head())


Получено статей: 40


,artist_key,artist_name,article_id,title,publication_date,author,body_text,word_count_api,tag_count,text_length,word_count,artist_mention_count,artist_in_title
0,post malone,Post Malone,music/2025/sep/21/post-malone-review-megawatt-charisma-and-anthemic-hooks-from-an-irresistibly-genial-outlaw,Post Malone review – megawatt charisma and anthemic hooks from an irresistibly genial outlaw,2025-09-21 11:19:49+00:00,Stevie Chick,post malone s been on quite the journey since he debuted a decade ago rapping over trap beats with a voice soaked in...,363,5,2160,375,1,True
1,post malone,Post Malone,music/2025/apr/14/post-malone-coachella-review,Post Malone at Coachella review – chameleonic megastar wows in the desert,2025-04-14 14:11:16+00:00,Adrian Horton,three songs into his coachella headliner set on sunday post malone asked his band to stop and restart it wasn t that...,739,6,4172,751,1,True
2,post malone,Post Malone,music/2025/nov/15/post-your-questions-for-peaches,Post your questions for Peaches,2025-11-15 12:04:32+00:00,Ben Beaumont-Thomas,whether crowdsurfing inside a giant condom or singing alongside a vulva-headed dancer peaches has left us with some ...,299,6,1636,299,0,False
3,post malone,Post Malone,music/2025/dec/04/post-your-questions-for-chrissie-hynde,Post your questions for Chrissie Hynde,2025-12-04 15:47:53+00:00,Ben Beaumont-Thomas,the guest list on chrissie hynde s latest album duets special is testament to the high esteem she s still held in de...,309,4,1713,313,0,False
4,post malone,Post Malone,music/2025/nov/28/htrk-string-of-hearts-songs-of-htrk-review-friends-from-liars-to-kali-malone-rework-their-noisy-gems,HTRK: String of Hearts (Songs of HTRK) review – friends from Liars to Kali Malone rework their noisy gems,2025-11-28 08:00:04+00:00,Safi Bugel,htrk have been making their gloomy sensual brand of music at the intersection of electronic pop and noise rock for 2...,470,3,2902,463,0,False


## 10. Векторизация текстов TF-IDF

Для преобразования текста в числовое представление используется TF-IDF. Это позволяет представить неструктурированные статьи в виде матрицы признаков и в дальнейшем использовать их в алгоритмах анализа данных.

Матрица хранится отдельно в разреженном формате, так как большинство значений TF-IDF равны нулю.

In [9]:
tfidf_matrix = sparse.csr_matrix((0, 0))
tfidf_feature_names = []

if not guardian_df.empty and guardian_df["body_text"].str.len().gt(0).any():
    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=100,
        min_df=1,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(guardian_df["body_text"])
    tfidf_feature_names = vectorizer.get_feature_names_out().tolist()

    print("Размер TF-IDF матрицы:", tfidf_matrix.shape)

    mean_weights = np.asarray(tfidf_matrix.mean(axis=0)).ravel()
    top_idx = mean_weights.argsort()[::-1][:15]

    tfidf_top = pd.DataFrame({
        "term": np.array(tfidf_feature_names)[top_idx],
        "mean_tfidf": mean_weights[top_idx]
    })
    display(tfidf_top)
else:
    print("Тексты Guardian пока отсутствуют — TF-IDF будет построен после задания API-ключа.")


Размер TF-IDF матрицы: (40, 100)


,term,mean_tfidf
0,music,0.096280
1,album,0.093426
2,sheeran,0.087541
3,year,0.082931
4,said,0.076357
5,eilish,0.075019
6,song,0.069419
7,like,0.068745
8,pop,0.068095
9,tour,0.066839


## 11. Агрегация текстовых признаков до уровня исполнителя

Spotify содержит много недельных наблюдений, а статьи Guardian являются отдельными документами. Для корректного слияния текстовые признаки агрегируются до уровня исполнителя.

In [10]:
if not guardian_df.empty:
    guardian_artist = (
        guardian_df.groupby("artist_key", as_index=False)
        .agg(
            article_count=("article_id", "nunique"),
            avg_article_words=("word_count", "mean"),
            total_artist_mentions=("artist_mention_count", "sum"),
            avg_text_length=("text_length", "mean"),
            avg_tag_count=("tag_count", "mean")
        )
    )
else:
    guardian_artist = pd.DataFrame(columns=[
        "artist_key", "article_count", "avg_article_words",
        "total_artist_mentions", "avg_text_length", "avg_tag_count"
    ])

display(guardian_artist.head())


,artist_key,article_count,avg_article_words,total_artist_mentions,avg_text_length,avg_tag_count
0,ariana grande,5,859.4,7,4810.8,7.0
1,bad bunny,5,771.6,14,4375.4,10.0
2,billie eilish,5,895.6,8,5200.0,10.0
3,drake,5,623.8,60,3470.2,6.8
4,ed sheeran,5,730.0,16,4110.2,7.0


## 12. Общие атрибуты для слияния

Источники имеют разные форматы, поэтому перед объединением определены общие агрегаты и технические ключи.

| Источники | Общие атрибуты |
|---|---|
| Spotify + Last.fm | artist_key + track_key |
| Spotify + MusicBrainz | artist_key, дополнительно mbid после сопоставления |
| Spotify + Guardian | artist_key после агрегации статей |
| Временной анализ | week, release_date, publication_date |

artist_key и track_key создаются путем приведения строк к нижнему регистру, удаления лишних символов и повторных пробелов. Это снижает количество несовпадений из-за различий в написании.

## 13. Слияние разнородных данных

Основой итогового набора остается очищенный Spotify. Дополнительные данные подключаются левым соединением, чтобы не терять основные недельные наблюдения. После каждого слияния рассчитывается покрытие — доля строк, для которых удалось получить внешний признак.

In [11]:
merged = spotify.copy()

# Last.fm: уровень трека
if not lastfm_df.empty:
    lastfm_unique = lastfm_df.drop_duplicates(
        subset=["artist_key", "track_key"],
        keep="last"
    )

    merged = merged.merge(
        lastfm_unique,
        on=["artist_key", "track_key"],
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "lfm_mbid", "lfm_duration", "listeners",
        "playcount", "lfm_album", "lfm_tags"
    ]:
        merged[col] = np.nan

# MusicBrainz: уровень исполнителя
if not musicbrainz_df.empty:
    mb_unique = musicbrainz_df.drop_duplicates("artist_key", keep="first")
    merged = merged.merge(
        mb_unique,
        on="artist_key",
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "mbid", "mb_type", "mb_name", "country",
        "gender", "active_begin", "active_end", "alias_count"
    ]:
        merged[col] = np.nan

# Guardian: уровень исполнителя
if not guardian_artist.empty:
    merged = merged.merge(
        guardian_artist,
        on="artist_key",
        how="left",
        validate="m:1"
    )
else:
    for col in [
        "article_count", "avg_article_words",
        "total_artist_mentions", "avg_text_length", "avg_tag_count"
    ]:
        merged[col] = np.nan

print("Итоговый размер:", merged.shape)

coverage = {}
for col in ["listeners", "mbid", "article_count"]:
    if col in merged.columns:
        coverage[col] = round(merged[col].notna().mean() * 100, 2)

display(pd.DataFrame(
    coverage.items(),
    columns=["external_feature", "coverage_%"]
))

display(merged.head())


Итоговый размер: (44195, 61)


,external_feature,coverage_%
0,listeners,8.74
1,mbid,18.67
2,article_count,16.17


,track_id,track_name,track_popularity,track_number,album_id,album_name,album_img,album_type,album_label,album_track_number,album_popularity,artist_num,artist_names,artist_id,artist_name,artist_img,artist_followers,artist_popularity,artist_genres,rank,week,collab,explicit,release_date,danceability,energy,key,mode,time_signature,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration,pivot,streams,track_index,artist_key,track_key,lfm_mbid,lfm_duration,listeners,playcount,lfm_album,lfm_tags,mbid,mb_type,mb_name,country,gender,active_begin,active_end,alias_count,article_count,avg_article_words,total_artist_mentions,avg_text_length,avg_tag_count
0,5aAx2yezTd8zXrkmtKl66Z,Starboy,0,1,09fggMHib4YkOtwQNXEBII,Starboy,https://i.scdn.co/image/ab67616d0000b2730c8599cbde51245c128bcea9,album,Universal Music Group,1,0,2,"The Weeknd, Daft Punk",1Xyo4u8uXC1ZmMpatF05PJ,The Weeknd,https://i.scdn.co/image/ab6761610000e5ebb5f9e28219c169fd4b9e8379,51215861,94,"canadian contemporary r&b, canadian pop, pop",1,2017-01-06,True,True,2016-11-25,0.681,0.594,7,True,4,-7.028,0.2820,0.1650,0.000003,0.134,0.5350,186.054,230453,False,25734078,1,the weeknd,starboy,NaN,NaN,NaN,NaN,NaN,NaN,c8b03190-306c-4120-bb0b-6f2ebfc06ea9,Person,The Weeknd,CA,male,1990-02-16,,7.0,5.0,1164.0,24.0,6243.0,6.6
1,7BKLCZ1jbUBVqRi2FVlTVw,Closer,84,1,0rSLgV8p5FzfnqlEk4GzxE,Closer,https://i.scdn.co/image/ab67616d0000b273495ce6da9aeb159e94eaa453,single,Disruptor Records/Columbia,1,71,2,"The Chainsmokers, Halsey",69GGBxA162lTqCwzJG5jLp,The Chainsmokers,https://i.scdn.co/image/ab6761610000e5eb3c02f4fb4cc9187c488afd50,19562506,81,"dance pop, edm, electropop, pop, pop dance, tropical house",2,2017-01-06,True,False,2016-07-29,0.748,0.524,8,True,4,-5.599,0.0338,0.4140,0.000000,0.111,0.6610,95.010,244960,False,23519705,2,the chainsmokers,closer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,5knuzwU65gJK7IF5yJsuaW,Rockabye (feat. Sean Paul & Anne-Marie),75,1,3meZFplbMmji648oWUNEfQ,Rockabye (feat. Sean Paul & Anne-Marie),https://i.scdn.co/image/ab67616d0000b2731431c3bdf16aa99f71799d95,single,Atlantic Records UK,1,62,3,"Clean Bandit, Sean Paul, Anne-Marie",6MDME20pz9RveH9rEXvrOM,Clean Bandit,https://i.scdn.co/image/ab6761610000e5eb7a487027eb0c10af725d5410,4943046,76,"dance pop, edm, pop, pop dance, post-teen pop, tropical house, uk dance, uk funky, uk pop",3,2017-01-06,True,False,2016-10-21,0.720,0.763,9,False,4,-4.068,0.0523,0.4060,0.000000,0.180,0.7420,101.965,251088,False,21216399,3,clean bandit,rockabye feat sean paul anne marie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4pdPtRcBmOSQDlJ3Fk945m,Let Me Love You,2,13,55bbXORm6ZrVq52zfZnxBf,Encore,https://i.scdn.co/image/ab67616d0000b2735045de6786e0af096de73ed4,album,Universal Music Group,13,0,2,"DJ Snake, Justin Bieber",540vIaP2JwjQb9dm3aArA4,DJ Snake,https://i.scdn.co/image/ab6761610000e5ebf2855e25f1d9c8a20bcc85ae,8137850,78,"dance pop, edm, electronic trap, pop, pop dance, pop rap",4,2017-01-06,True,False,2016-08-05,0.476,0.718,8,True,4,-5.309,0.0576,0.0784,0.000010,0.122,0.1420,199.864,205946,False,19852704,4,dj snake,let me love you,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3NdDpSvN911VPGivFlV5d0,"I Don’t Wanna Live Forever (Fifty Shades Darker) - From ""Fifty Shades Darker (Original Motion Picture Soundtrack)""",0,1,5MxXY7DbFMUiHFTPUabgJJ,I Don’t Wanna Live Forever (Fifty Shades Darker),https://i.scdn.co/image/ab67616d0000b2732ae921f082ae5db4d3da818f,single,Universal Music Group,1,0,2,"ZAYN, Taylor Swift",5ZsFI1h6hIdQRw2ti0hz81,ZAYN,https://i.scdn.co/image/ab6761610000e5eb9fbf7133dfc04d4cd44ccd36,21433236,77,"dance pop, pop, post-teen pop, uk pop",5,2017-01-06,True,False,2016-12-09,0.735,0.451,0,True,4,-8.374,0.0585,0.0631,0.000013,0.325,0.0862,117.973,245200,False,18316326,5,zayn,i don t wanna live forever fifty shades darker from fifty shades darker original motion picture soundtrack,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

## 14. Контроль качества после слияния

Проверим пропуски, дубликаты ключевых наблюдений и базовые статистики итогового набора. Пропуски во внешних API-признаках не удаляются: они означают, что для конкретного объекта внешний источник не дал совпадение. Это важно отличать от ошибки в основном наборе.

In [12]:
quality = pd.DataFrame({
    "dtype": merged.dtypes.astype(str),
    "missing": merged.isna().sum(),
    "missing_%": (merged.isna().mean() * 100).round(2),
    "unique": merged.nunique(dropna=True)
}).sort_values("missing_%", ascending=False)

display(quality.head(30))

key_cols = [c for c in ["track_id", "week"] if c in merged.columns]
if key_cols:
    print(
        "Дубликатов по ключу",
        key_cols,
        ":",
        merged.duplicated(subset=key_cols).sum()
    )

summary_cols = [
    c for c in [
        "streams", "rank", "track_popularity", "artist_popularity",
        "artist_followers", "danceability", "energy", "valence",
        "tempo", "duration", "listeners", "playcount"
    ]
    if c in merged.columns
]

display(merged[summary_cols].describe().T)


,dtype,missing,missing_%,unique
lfm_duration,float64,40334,91.26,30
lfm_mbid,object,40334,91.26,30
listeners,float64,40334,91.26,30
playcount,float64,40334,91.26,30
lfm_album,object,40334,91.26,26
lfm_tags,object,40334,91.26,28
avg_tag_count,float64,37048,83.83,5
article_count,float64,37048,83.83,1
avg_article_words,float64,37048,83.83,8
total_artist_mentions,float64,37048,83.83,8


Дубликатов по ключу ['track_id', 'week'] : 0


,count,mean,std,min,25%,50%,75%,max
streams,44195.0,8.349889e+06,5.824783e+06,2.525159e+06,4921144.500,6.289780e+06,9.612280e+06,8.076404e+07
rank,44195.0,1.005087e+02,5.773218e+01,1.000000e+00,51.000,1.010000e+02,1.510000e+02,2.000000e+02
track_popularity,44195.0,5.808035e+01,3.013181e+01,0.000000e+00,56.000,7.100000e+01,7.800000e+01,9.200000e+01
artist_popularity,44195.0,8.098375e+01,9.150428e+00,0.000000e+00,76.000,8.200000e+01,8.800000e+01,1.000000e+02
artist_followers,44195.0,2.348336e+07,2.476296e+07,1.000000e+01,5258096.000,1.391340e+07,3.681735e+07,1.040598e+08
danceability,44195.0,6.995385e-01,1.326356e-01,1.500000e-01,0.622,7.190000e-01,7.920000e-01,9.800000e-01
energy,44195.0,6.333073e-01,1.599436e-01,2.790000e-02,0.530,6.480000e-01,7.570000e-01,9.780000e-01
valence,44195.0,5.026141e-01,2.205627e-01,3.200000e-02,0.337,4.940000e-01,6.760000e-01,9.820000e-01
tempo,44195.0,1.202131e+02,2.926070e+01,4.671800e+01,96.053,1.167350e+02,1.400220e+02,2.121170e+02
duration,44195.0,2.069923e+05,4.083393e+04,3.013300e+04,181909.500,2.043460e+05,2.274760e+05,5.771990e+05


## 15. Выбор структуры хранения

Для объединенного табличного набора выбран формат **Parquet**.

Почему он подходит:
1. сохраняет типы данных лучше CSV;
2. использует сжатое колоночное хранение;
3. удобен для последующего анализа отдельных признаков;
4. быстрее загружается при повторной работе с крупными таблицами;
5. поддерживается pandas и другими инструментами обработки больших данных.

TF-IDF является разреженной матрицей, поэтому она сохраняется отдельно в формате NPZ. Список признаков TF-IDF сохраняется в JSON.

Исходные данные из API не заменяют основной набор, а выступают слоями обогащения. Такая структура позволяет повторно обновлять отдельный источник без полной перестройки всех исходных данных.

In [13]:
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_parquet_or_csv(df, stem):
    parquet_path = OUTPUT_DIR / f"{stem}.parquet"
    try:
        df.to_parquet(parquet_path, index=False)
        print("Сохранено:", parquet_path)
        return parquet_path
    except Exception as exc:
        csv_path = OUTPUT_DIR / f"{stem}.csv"
        df.to_csv(csv_path, index=False)
        print("Parquet недоступен:", exc)
        print("Резервно сохранено:", csv_path)
        return csv_path

save_parquet_or_csv(spotify, "spotify_clean")
save_parquet_or_csv(musicbrainz_df, "musicbrainz_artists")
save_parquet_or_csv(lastfm_df, "lastfm_tracks")
save_parquet_or_csv(guardian_df, "guardian_articles")
save_parquet_or_csv(merged, "music_merged")

if tfidf_matrix.shape[0] > 0:
    sparse.save_npz(OUTPUT_DIR / "guardian_tfidf.npz", tfidf_matrix)

    with open(
        OUTPUT_DIR / "guardian_tfidf_features.json",
        "w",
        encoding="utf-8"
    ) as file:
        json.dump(tfidf_feature_names, file, ensure_ascii=False, indent=2)


Parquet недоступен: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
Резервно сохранено: data/processed/spotify_clean.csv
Parquet недоступен: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

## 16. Скрипт обновления подготовленных данных

Для дальнейших этапов нужна возможность дополнять уже сохраненные данные новыми API-ответами. Используется функция upsert: новые строки добавляются, а записи с совпадающими ключами заменяются более свежей версией.

In [14]:
def upsert_dataframe(existing, new_data, keys):
    """
    Объединяет старую и новую версии набора.
    При совпадении ключей сохраняется последняя (новая) запись.
    """
    if existing is None or existing.empty:
        return new_data.copy()

    if new_data is None or new_data.empty:
        return existing.copy()

    result = pd.concat(
        [existing, new_data],
        ignore_index=True,
        sort=False
    )

    return result.drop_duplicates(
        subset=keys,
        keep="last"
    ).reset_index(drop=True)

# Пример обновления Last.fm:
lastfm_updated = upsert_dataframe(
    lastfm_df,
    pd.DataFrame(columns=lastfm_df.columns),
    keys=["artist_key", "track_key"]
)

print("Размер Last.fm после тестового обновления:", lastfm_updated.shape)


Размер Last.fm после тестового обновления: (30, 8)


## 17. Итог этапа 2.3

На этапе подготовки:

- загружен и исследован основной реляционный набор Spotify;
- проверены размер, типы данных, пропуски и дубликаты;
- описан физический смысл и аналитическая роль основных атрибутов;
- признаки распределены по восьми типам аналитических шкал;
- выполнена очистка Spotify и нормализация ключей;
- реализован разбор XML MusicBrainz;
- реализована загрузка и подготовка JSON Last.fm;
- реализованы сбор, очистка и аннотирование текстов The Guardian;
- выполнена TF-IDF-векторизация неструктурированных данных;
- определены общие атрибуты и агрегаты разных источников;
- реализовано слияние данных с контролем покрытия;
- выбран Parquet для хранения объединенных таблиц и NPZ для разреженной TF-IDF матрицы;
- подготовлена функция обновления данных по ключам.

Итоговый набор music_merged предназначен для следующего этапа 2.4, где будут применяться алгоритмы анализа данных и машинного обучения для проверки аналитической гипотезы.

**Важно:** для фактического заполнения блоков Last.fm и The Guardian перед финальным запуском необходимо задать соответствующие API-ключи и выполнить ноутбук сверху вниз. Секретные ключи в публичный репозиторий не сохраняются.